1. 讀取資料

In [1]:
import geopandas as gpd

# 讀取本地的河川資料
rivers = gpd.read_file('riverpoly/riverpoly.shp')
print(f"河川資料載入完成，共 {len(rivers)} 筆")
print(f"CRS: {rivers.crs}")

import pandas as pd

# 讀取避難所資料
shelters_csv = pd.read_csv('避難收容處所點位檔案v9.csv')
print(f"避難所資料載入完成，共 {len(shelters_csv)} 筆")

# 轉換為 GeoDataFrame
shelters = gpd.GeoDataFrame(
    shelters_csv,
    geometry=gpd.points_from_xy(shelters_csv['經度'], shelters_csv['緯度']),
    crs='EPSG:4326'
)
print(f"避難所 GeoDataFrame 資料載入完成，共 {len(shelters)} 筆")

# 讀取鄉鎮資料 - 指定正確的圖層
from urllib.parse import quote
url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')
print(f"鄉鎮資料載入完成，共 {len(townships)} 筆")

河川資料載入完成，共 13262 筆
CRS: EPSG:3826
避難所資料載入完成，共 5973 筆
避難所 GeoDataFrame 資料載入完成，共 5973 筆
鄉鎮資料載入完成，共 368 筆


2. 確認CRS

In [2]:
# CRS 轉換 - 確保所有資料都是 EPSG:3826

# 1. 確認 rivers 是否為 EPSG:3826
print(f"河川原始 CRS: {rivers.crs}")
if rivers.crs.to_epsg() == 3826:
    print("✓ 河川資料已經是 EPSG:3826")
else:
    print("⚠ 河川資料不是 EPSG:3826，正在轉換...")
    rivers = rivers.to_crs('EPSG:3826')
    print(f"✓ 河川資料已轉換為 EPSG:3826")

# 2. 將 shelter 轉換為 EPSG:3826
print(f"避難所原始 CRS: {shelters.crs}")
shelters = shelters.to_crs('EPSG:3826')
print(f"✓ 避難所資料已轉換為 EPSG:3826")
print(f"避難所轉換後 CRS: {shelters.crs}")

# 3. 將 townships 轉換為 EPSG:3826
print(f"鄉鎮原始 CRS: {townships.crs}")
townships = townships.to_crs('EPSG:3826')
print(f"✓ 鄉鎮資料已轉換為 EPSG:3826")
print(f"鄉鎮轉換後 CRS: {townships.crs}")

print("\n 所有資料都已轉換為 EPSG:3826，可以進行後續分析！")

河川原始 CRS: EPSG:3826
✓ 河川資料已經是 EPSG:3826
避難所原始 CRS: EPSG:4326
✓ 避難所資料已轉換為 EPSG:3826
避難所轉換後 CRS: EPSG:3826
鄉鎮原始 CRS: GEOGCS["GCS_TWD97[2020]",DATUM["Taiwan_Datum_1997",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1026"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]]
✓ 鄉鎮資料已轉換為 EPSG:3826
鄉鎮轉換後 CRS: EPSG:3826

 所有資料都已轉換為 EPSG:3826，可以進行後續分析！


3. 資料清理

In [9]:
# 避難所資料清理 - 結合座標範圍和鄉鎮界線檢查

print("=== 避難所資料清理 ===")

# 重新從原始 CSV 讀取資料
import pandas as pd
import geopandas as gpd

shelters_csv = pd.read_csv('避難收容處所點位檔案v9.csv')
print(f"重新讀取原始 CSV，共 {len(shelters_csv)} 筆")

# 重新建立 GeoDataFrame
shelters_original = gpd.GeoDataFrame(
    shelters_csv,
    geometry=gpd.points_from_xy(shelters_csv['經度'], shelters_csv['緯度']),
    crs='EPSG:4326'
)

# 轉換為 EPSG:3826
shelters_original = shelters_original.to_crs('EPSG:3826')

print(f"重新建立 GeoDataFrame，共 {len(shelters_original)} 筆")

original_count = len(shelters_original)

# === 第一步：座標範圍檢查 ===
print(f"\n=== 第一步：座標範圍檢查 ===")

# 1. 檢查空值和零值的筆數
null_lon = shelters_original['經度'].isna().sum()
null_lat = shelters_original['緯度'].isna().sum()
zero_lon = (shelters_original['經度'] == 0).sum()
zero_lat = (shelters_original['緯度'] == 0).sum()

print(f"經度為空值: {null_lon} 筆")
print(f"緯度為空值: {null_lat} 筆")
print(f"經度為 0: {zero_lon} 筆")
print(f"緯度為 0: {zero_lat} 筆")

# 2. 檢查超出台灣範圍的筆數 (經度 119~122, 緯度 21~26)
out_of_range_condition = (
    (shelters_original['經度'] < 119) | (shelters_original['經度'] > 122) |
    (shelters_original['緯度'] < 21) | (shelters_original['緯度'] > 26)
)

out_of_range_count = out_of_range_condition.sum()
print(f"超出台灣範圍(經度119~122, 緯度21~26): {out_of_range_count} 筆")

# 3. 座標範圍清理
range_clean_condition = (
    shelters_original['經度'].notna() &
    shelters_original['緯度'].notna() &
    (shelters_original['經度'] != 0) &
    (shelters_original['緯度'] != 0) &
    (shelters_original['經度'] >= 119) &
    (shelters_original['經度'] <= 122) &
    (shelters_original['緯度'] >= 21) &
    (shelters_original['緯度'] <= 26)
)

shelters_step1 = shelters_original[range_clean_condition]
range_removed = original_count - len(shelters_step1)

print(f"座標範圍清理後剩餘: {len(shelters_step1)} 筆")
print(f"座標範圍移除: {range_removed} 筆")

# === 第二步：鄉鎮界線檢查 ===
print(f"\n=== 第二步：鄉鎮界線檢查 ===")

# 確保 CRS 一致
if shelters_step1.crs != townships.crs:
    print("CRS 不一致，正在轉換...")
    shelters_step1 = shelters_step1.to_crs(townships.crs)

# 使用 spatial join 找出在鄉鎮界線內的避難所
shelters_in_townships = gpd.sjoin(shelters_step1, townships, how='inner', predicate='within')
township_removed = len(shelters_step1) - len(shelters_in_townships)

print(f"在鄉鎮界線內: {len(shelters_in_townships)} 筆")
print(f"不在鄉鎮界線內: {township_removed} 筆")

# === 最終清理結果 ===
shelter_true = shelters_in_townships.copy()

# 移除 sjoin 產生的多餘欄位
if 'index_right' in shelter_true.columns:
    shelter_true = shelter_true.drop(columns=['index_right'])

final_count = len(shelter_true)
total_removed = original_count - final_count

print(f"\n=== 最終清理結果 ===")
print(f"清理前資料筆數: {original_count}")
print(f"清理後資料筆數: {final_count}")
print(f"總移除: {total_removed} 筆")
print(f"資料保留率: {final_count/original_count*100:.1f}%")

print(f"\n移除原因分析：")
print(f"- 座標範圍問題: {range_removed} 筆")
print(f"- 不在鄉鎮界線內: {township_removed} 筆")
print(f"- 總計: {total_removed} 筆")

# 檢查清理後的座標範圍
print(f"\n清理後座標範圍:")
print(f"經度範圍: {shelter_true['經度'].min():.6f} ~ {shelter_true['經度'].max():.6f}")
print(f"緯度範圍: {shelter_true['緯度'].min():.6f} ~ {shelter_true['緯度'].max():.6f}")

# 統計各縣市的避難所數量
if 'COUNTYNAME' in shelter_true.columns:
    print(f"\n各縣市避難所分布:")
    county_counts = shelter_true['COUNTYNAME'].value_counts().head(10)
    for county, count in county_counts.items():
        print(f"  - {county}: {count} 筆")

print(f"\n✓ 避難所資料清理完成，shelter_true 已更新為清理後的 {len(shelter_true)} 筆資料")
print("✓ 結合座標範圍檢查和鄉鎮界線檢查，確保資料品質")

=== 避難所資料清理 ===
重新讀取原始 CSV，共 5973 筆
重新建立 GeoDataFrame，共 5973 筆

=== 第一步：座標範圍檢查 ===
經度為空值: 0 筆
緯度為空值: 0 筆
經度為 0: 3 筆
緯度為 0: 0 筆
超出台灣範圍(經度119~122, 緯度21~26): 85 筆
座標範圍清理後剩餘: 5888 筆
座標範圍移除: 85 筆

=== 第二步：鄉鎮界線檢查 ===
在鄉鎮界線內: 5859 筆
不在鄉鎮界線內: 29 筆

=== 最終清理結果 ===
清理前資料筆數: 5973
清理後資料筆數: 5859
總移除: 114 筆
資料保留率: 98.1%

移除原因分析：
- 座標範圍問題: 85 筆
- 不在鄉鎮界線內: 29 筆
- 總計: 114 筆

清理後座標範圍:
經度範圍: 119.425916 ~ 121.988800
緯度範圍: 21.991200 ~ 25.978280

各縣市避難所分布:
  - 新北市: 542 筆
  - 臺南市: 535 筆
  - 高雄市: 504 筆
  - 臺中市: 479 筆
  - 桃園市: 435 筆
  - 彰化縣: 418 筆
  - 南投縣: 389 筆
  - 嘉義縣: 371 筆
  - 雲林縣: 327 筆
  - 臺北市: 311 筆

✓ 避難所資料清理完成，shelter_true 已更新為清理後的 5859 筆資料
✓ 結合座標範圍檢查和鄉鎮界線檢查，確保資料品質
